# 使用 OpenVINO 创建基于大语言模型的聊天机器人

在人工智能（AI）快速发展的世界中，聊天机器人已成为企业增强客户互动和优化运营的强大工具。  
大语言模型（LLMs）是能够理解并生成人类语言的人工智能系统。它们利用深度学习算法和海量数据，学习语言的细微差别，并生成连贯且相关的内容。  
虽然基于意图的聊天机器人可以回答订单管理、常见问题（FAQ）和政策咨询等基本的一次性查询，但基于LLM的聊天机器人能够处理更复杂、多轮次的对话问题。LLM使聊天机器人能够通过上下文记忆，以类似人类的方式进行对话式支持。借助语言模型的能力，聊天机器人正变得越来越智能，能够以极高的准确性理解并回应人类语言。

之前，我们已经讨论了如何使用 OpenVINO 和 Optimum Intel 构建指令遵循管道，请参考 [Dolly 示例](../dolly-2-instruction-following)。  
在本教程中，我们将探讨如何利用 OpenVINO 的能力运行大语言模型以实现聊天功能。我们将使用来自 [Hugging Face Transformers](https://huggingface.co/docs/transformers/index) 库的预训练模型。为简化用户体验，使用 [Hugging Face Optimum Intel](https://huggingface.co/docs/optimum/intel/index) 库将模型转换为 OpenVINO™ IR 格式并创建推理管道。此外，也可以使用 [OpenVINO Generate API](https://github.com/openvinotoolkit/openvino.genai/tree/master/src) 创建推理管道，相关示例请参见笔记本 [LLM chatbot with OpenVINO Generate API](./llm-chatbot-generate-api.ipynb)。

本教程包含以下步骤：

- 安装先决条件  
- 使用 [OpenVINO 与 Hugging Face Optimum 的集成](https://huggingface.co/blog/openvino) 从公开源下载并转换模型  
- 使用 [NNCF](https://github.com/openvinotoolkit/nncf) 将模型权重压缩为 4 位或 8 位数据类型  
- 创建聊天推理管道  
- 运行聊天管道  

#### 目录：

- [先决条件](#Prerequisites)  
- [选择推理模型](#Select-model-for-inference)  
- [使用 Optimum-CLI 工具转换模型](#Convert-model-using-Optimum-CLI-tool)  
- [压缩模型权重](#Compress-model-weights)  
    - [使用 Optimum-CLI 压缩权重](#Weights-Compression-using-Optimum-CLI)  
    - [使用 AWQ 压缩权重](#Weight-compression-with-AWQ)  
- [选择推理设备和模型变体](#Select-device-for-inference-and-model-variant)  
- [使用 Optimum Intel 实例化模型](#Instantiate-Model-using-Optimum-Intel)  
- [运行聊天机器人](#Run-Chatbot)  

### 安装说明

这是一个自包含的示例，仅依赖于其自身的代码。

我们建议在虚拟环境中运行此笔记本。您只需一个 Jupyter 服务器即可开始。  
更多详情，请参阅 [安装指南](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide)。

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/llm-chatbot/llm-chatbot.ipynb" />

## 前提条件
[返回顶部 ⬆️](#Table-of-contents:)

安装所需的依赖项

In [ ]:
import os
import platform

os.environ["GIT_CLONE_PROTECTION_ACTIVE"] = "false"

%pip install -Uq pip
%pip uninstall -q -y optimum optimum-intel
%pip install --pre -Uq "openvino>=2025.3.0" openvino-tokenizers[transformers] --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly
%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu\
"git+https://github.com/huggingface/optimum-intel.git"\
"nncf>=2.18.0"\
"torch==2.8" \
"datasets<4.0.0" \
"accelerate" \
"gradio>=4.19" \
"huggingface-hub>=0.26.5" \
 "einops" "transformers==4.53.3" "transformers_stream_generator" "tiktoken" "bitsandbytes"

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0.0"

In [ ]:
import os
from pathlib import Path
import requests
import shutil

# 获取模型配置

config_shared_path = Path("../../utils/llm_config.py")
config_dst_path = Path("llm_config.py")

if not config_dst_path.exists():
    if config_shared_path.exists():
        try:
            os.symlink(config_shared_path, config_dst_path)
        except Exception:
            shutil.copy(config_shared_path, config_dst_path)
    else:
        r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/llm_config.py")
        with open("llm_config.py", "w", encoding="utf-8") as f:
            f.write(r.text)
elif not os.path.islink(config_dst_path):
    print("LLM config will be updated")
    if config_shared_path.exists():
        shutil.copy(config_shared_path, config_dst_path)
    else:
        r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/llm_config.py")
        with open("llm_config.py", "w", encoding="utf-8") as f:
            f.write(r.text)

## 选择用于推理的模型
[返回顶部 ⬆️](#Table-of-contents:)

本教程支持多种模型，您可以从提供的选项中选择一个进行比较，以评估开源大语言模型解决方案的质量。
>**注意**：某些模型的转换可能需要用户额外操作，并且至少需要64GB内存进行转换。

<details>
  <summary><b>点击此处查看可用模型选项</b></summary>

* **tiny-llama-1b-chat** - 这是基于 [TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T](https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T) 微调得到的对话模型。TinyLlama项目旨在使用与Llama 2相同的架构和分词器，在3万亿token上预训练一个11亿参数的Llama模型。这意味着TinyLlama可以无缝集成到许多基于Llama构建的开源项目中。此外，TinyLlama体积紧凑，仅有11亿参数，这种紧凑性使其适用于对计算和内存资源有限制的各种应用场景。更多模型细节请参见 [模型卡片](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0)
*  **minicpm-2b-dpo** - MiniCPM是由ModelBest Inc.和清华大学NLP团队开发的端侧大语言模型，不含嵌入层共24亿参数。经过直接偏好优化（DPO）微调后，MiniCPM在多个70亿、130亿和700亿参数的主流模型中表现更优。更多细节请参见 [模型卡片](https://huggingface.co/openbmb/MiniCPM-2B-dpo-fp16)。
*  **minicpm3-4b** - MiniCPM3-4B是MiniCPM系列的第三代产品。整体性能超越Phi-3.5-mini-Instruct，与近期多数70亿至90亿参数模型相当。相比前代，MiniCPM3-4B具备更强大和多样化的技能集，可满足更广泛的应用需求。更多细节请参见 [模型卡片](https://huggingface.co/openbmb/MiniCPM3-4B)。
* **minicpm4-8b** - MiniCPM 4是一个极致高效的边缘侧大模型，在模型架构、学习算法、训练数据和推理系统四个维度进行了高效优化，实现了极致的效率提升。[模型卡片](https://huggingface.co/openbmb/MiniCPM4-8B)。
*  **gemma-2b-it** - Gemma是由谷歌推出的轻量级、最先进的开源模型家族，基于创建Gemini模型的研究和技术打造。它们是文本到文本的解码器-only 大语言模型，支持英文，拥有开放权重、预训练变体和指令微调变体。Gemma模型非常适合多种文本生成任务，包括问答、摘要和推理。此模型为20亿参数模型的指令微调版本。更多详情请参见 [模型卡片](https://huggingface.co/google/gemma-2b-it)。
>**注意**：运行带有演示的模型时，您需要接受许可协议。  
>您必须是🤗 Hugging Face Hub的注册用户。请访问 [HuggingFace模型卡片](https://huggingface.co/google/gemma-2b-it)，仔细阅读使用条款并点击接受按钮。您将需要使用访问令牌来运行以下代码。有关访问令牌的更多信息，请参考 [文档中的此部分](https://huggingface.co/docs/hub/security-tokens)。  
>您可以在笔记本环境中通过以下代码登录Hugging Face Hub：

```python
    ## 登录Hugging Face Hub以获取预训练模型访问权限

    from huggingface_hub import notebook_login, whoami

    try:
        whoami()
        print('授权令牌已提供')
    except OSError:
        notebook_login()
```
*  **gemma-2-2b-it** - Gemma2是谷歌推出的轻量级、最先进的开源模型家族的第二代，基于与创建Gemini模型相同的研究和技术打造。它们是文本到文本的解码器-only 大语言模型，支持英文，拥有开放权重、预训练变体和指令微调变体。Gemma模型非常适合多种文本生成任务，包括问答、摘要和推理。此模型为20亿参数模型的指令微调版本。更多详情请参见 [模型卡片](https://huggingface.co/google/gemma-2-2b-it)。
>**注意**：运行带有演示的模型时，您需要接受许可协议。  
>您必须是🤗 Hugging Face Hub的注册用户。请访问 [HuggingFace模型卡片](https://huggingface.co/google/gemma-2-2b-it)，仔细阅读使用条款并点击接受按钮。您将需要使用访问令牌来运行以下代码。有关访问令牌的更多信息，请参考 [文档中的此部分](https://huggingface.co/docs/hub/security-tokens)。  
>您可以在笔记本环境中通过以下代码登录Hugging Face Hub：

```python
    # 登录Hugging Face Hub以获取预训练模型访问权限

    from huggingface_hub import notebook_login, whoami

    try:
        whoami()
        print('授权令牌已提供')
    except OSError:
        notebook_login()
```
* **phi-3-mini-instruct** - Phi-3-Mini是一个38亿参数的轻量级、最先进的开源模型，使用Phi-3数据集进行训练，该数据集包含合成数据和经过筛选的公开网站数据，重点关注高质量和推理密集特性。更多模型详情请参见 [模型卡片](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct)、[微软博客](https://aka.ms/phi3blog-april) 和 [技术报告](https://aka.ms/phi3-tech-report)。
* **phi-3.5-mini-instruct** - Phi-3.5-mini是一个轻量级、最先进的开源模型，基于Phi-3使用的数据集——合成数据和筛选后的公开网站数据——专注于高质量、推理密集的数据。该模型属于Phi-3模型家族，支持128K token上下文长度。该模型经历了严格的增强过程，结合了监督微调、近端策略优化和直接偏好优化，以确保精确的指令遵循和稳健的安全措施。更多模型详情请参见 [模型卡片](https://huggingface.co/microsoft/Phi-3.5-mini-instruct)、[微软博客](https://aka.ms/phi3.5-techblog) 和 [技术报告](https://arxiv.org/abs/2404.14219)。
* **phi-4-mini-instruct** - Phi-4-mini是一个轻量级、最先进的开源模型，基于合成数据集和筛选后的公共领域网站数据混合构建，专注于高质量、密集推理数据。更多模型详情请参见 [模型卡片](https://huggingface.co/microsoft/Phi-4-mini-instruct)。
* **phi-4** - Phi-4是一个140亿参数的模型，基于合成数据集、筛选后的公共领域网站数据以及获取的学术书籍和问答数据集混合构建。该方法的目标是确保小规模能力模型在高质量和高级推理数据上进行训练。Phi-4经历了严格的增强和对齐过程，结合了监督微调和直接偏好优化，以确保精确的指令遵循和稳健的安全措施。更多模型详情请参见 [模型卡片](https://huggingface.co/microsoft/phi-4)、[技术报告](https://arxiv.org/pdf/2412.08905) 和 [微软博客](https://techcommunity.microsoft.com/blog/aiplatformblog/introducing-phi-4-microsoft%E2%80%99s-newest-small-language-model-specializing-in-comple/4357090)。
* **phi-4-mini-reasoning** - Phi-4-mini-reasoning是一个轻量级开源模型，基于合成数据构建，专注于高质量、推理密集数据，并进一步微调以获得更高级的数学推理能力。更多模型详情请参见 [模型卡片](https://huggingface.co/microsoft/Phi-4-mini-reasoning)。
* **phi-4-reasoning** - Phi-4-reasoning是一个最先进的开源权重推理模型，从Phi-4微调而来，使用链式思维轨迹数据集进行监督微调和强化学习。监督微调数据集包括合成提示和高质量筛选后的公共领域网站数据，重点在于数学、科学和编码技能以及安全和负责任AI的对齐数据。该方法的目标是确保小规模能力模型在高质量和高级推理数据上进行训练。更多模型详情请参见 [模型卡片](https://huggingface.co/microsoft/Phi-4-reasoning)。
* **red-pajama-3b-chat** - 基于GPT-NEOX架构的28亿参数预训练语言模型，由Together Computer和开源AI社区领导者共同开发。该模型在OASST1和Dolly2数据集上微调，以增强聊天能力。更多模型详情请参见 [HuggingFace模型卡片](https://huggingface.co/togethercomputer/RedPajama-INCITE-Chat-3B-v1)。
*  **gemma-7b-it** - Gemma是由谷歌推出的轻量级、最先进的开源模型家族，基于与创建Gemini模型相同的研究和技术打造。它们是文本到文本的解码器-only 大语言模型，支持英文，拥有开放权重、预训练变体和指令微调变体。Gemma模型非常适合多种文本生成任务，包括问答、摘要和推理。此模型为70亿参数模型的指令微调版本。更多详情请参见 [模型卡片](https://huggingface.co/google/gemma-7b-it)。
>**注意**：运行带有演示的模型时，您需要接受许可协议。  
>您必须是🤗 Hugging Face Hub的注册用户。请访问 [HuggingFace模型卡片](https://huggingface.co/google/gemma-7b-it)，仔细阅读使用条款并点击接受按钮。您将需要使用访问令牌来运行以下代码。有关访问令牌的更多信息，请参考 [文档中的此部分](https://huggingface.co/docs/hub/security-tokens)。  
>您可以在笔记本环境中通过以下代码登录Hugging Face Hub：

```python
    ## 登录Hugging Face Hub以获取预训练模型访问权限

    from huggingface_hub import notebook_login, whoami

    try:
        whoami()
        print('授权令牌已提供')
    except OSError:
        notebook_login()
```

*  **gemma-2-9b-it** - Gemma2是谷歌推出的轻量级、最先进的开源模型家族的第二代，基于与创建Gemini模型相同的研究和技术打造。它们是文本到文本的解码器-only 大语言模型，支持英文，拥有开放权重、预训练变体和指令微调变体。Gemma模型非常适合多种文本生成任务，包括问答、摘要和推理。此模型为90亿参数模型的指令微调版本。更多详情请参见 [模型卡片](https://huggingface.co/google/gemma-2-9b-it)。
>**注意**：运行带有演示的模型时，您需要接受许可协议。  
>您必须是🤗 Hugging Face Hub的注册用户。请访问 [HuggingFace模型卡片](https://huggingface.co/google/gemma-2-9b-it)，仔细阅读使用条款并点击接受按钮。您将需要使用访问令牌来运行以下代码。有关访问令牌的更多信息，请参考 [文档中的此部分](https://huggingface.co/docs/hub/security-tokens)。  
>您可以在笔记本环境中通过以下代码登录Hugging Face Hub：

```python
    # 登录Hugging Face Hub以获取预训练模型访问权限

    from huggingface_hub import notebook_login, whoami

    try:
        whoami()
        print('授权令牌已提供')
    except OSError:
        notebook_login()
```

* **llama-2-7b-chat** - LLama 2是由Meta开发的LLama模型的第二代。Llama 2是一组预训练和微调的生成文本模型，规模从70亿到700亿参数不等。llama-2-7b-chat是LLama 2的70亿参数版本，针对对话场景进行了微调和优化。更多模型详情请参见 [论文](https://ai.meta.com/research/publications/llama-2-open-foundation-and-fine-tuned-chat-models/)、[仓库](https://github.com/facebookresearch/llama) 和 [HuggingFace模型卡片](https://huggingface.co/meta-llama/Llama-2-7b-chat-hf)。
>**注意**：运行带有演示的模型时，您需要接受许可协议。  
>您必须是🤗 Hugging Face Hub的注册用户。请访问 [HuggingFace模型卡片](https://huggingface.co/meta-llama/Llama-2-7b-chat-hf)，仔细阅读使用条款并点击接受按钮。您将需要使用访问令牌来运行以下代码。有关访问令牌的更多信息，请参考 [文档中的此部分](https://huggingface.co/docs/hub/security-tokens)。  
>您可以在笔记本环境中通过以下代码登录Hugging Face Hub：

```python
    ## 登录Hugging Face Hub以获取预训练模型访问权限

    from huggingface_hub import notebook_login, whoami

    try:
        whoami()
        print('授权令牌已提供')
    except OSError:
        notebook_login()
```
* **llama-3-8b-instruct** - Llama 3是一种自回归语言模型，使用优化的Transformer架构。微调版本使用监督微调（SFT）和人类反馈强化学习（RLHF）来对齐人类偏好，以提高有用性和安全性。Llama 3指令微调模型针对对话场景优化，在常见行业基准测试中优于许多现有的开源聊天模型。更多模型详情请参见 [Meta博客文章](https://ai.meta.com/blog/meta-llama-3/)、[模型网站](https://llama.meta.com/llama3) 和 [模型卡片](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)。
>**注意**：运行带有演示的模型时，您需要接受许可协议。  
>您必须是🤗 Hugging Face Hub的注册用户。请访问 [HuggingFace模型卡片](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)，仔细阅读使用条款并点击接受按钮。您将需要使用访问令牌来运行以下代码。有关访问令牌的更多信息，请参考 [文档中的此部分](https://huggingface.co/docs/hub/security-tokens)。  
>您可以在笔记本环境中通过以下代码登录Hugging Face Hub：

```python
    ## 登录Hugging Face Hub以获取预训练模型访问权限

    from huggingface_hub import notebook_login, whoami

    try:
        whoami()
        print('授权令牌已提供')
    except OSError:
        notebook_login()
```
* **llama-3.1-8b-instruct** - Llama 3.1指令微调文本模型（8B、70B、405B）针对多语言对话场景优化，在常见行业基准测试中优于许多现有的开源和闭源聊天模型。更多模型详情请参见 [Meta博客文章](https://ai.meta.com/blog/meta-llama-3-1/)、[模型网站](https://llama.meta.com) 和 [模型卡片](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct)。
>**注意**：运行带有演示的模型时，您需要接受许可协议。  
>您必须是🤗 Hugging Face Hub的注册用户。请访问 [HuggingFace模型卡片](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct)，仔细阅读使用条款并点击接受按钮。您将需要使用访问令牌来运行以下代码。有关访问令牌的更多信息，请参考 [文档中的此部分](https://huggingface.co/docs/hub/security-tokens)。  
>您可以在笔记本环境中通过以下代码登录Hugging Face Hub：

```python
    ## 登录Hugging Face Hub以获取预训练模型访问权限

    from huggingface_hub import notebook_login, whoami

    try:
        whoami()
        print('授权令牌已提供')
    except OSError:
        notebook_login()
```

* **qwen2.5-0.5b-instruct/qwen2.5-1.5b-instruct/qwen2.5-3b-instruct/qwen2.5-7b-instruct/qwen2.5-14b-instruct** - Qwen2.5是Qwen大型语言模型的最新系列。相较于Qwen2，Qwen2.5系列在编码、数学和通用知识技能方面有显著改进。此外，它还引入了长上下文和多语言支持，包括中文、英文、法语、西班牙语、葡萄牙语、德语、意大利语、俄语、日语、韩语、越南语、泰语、阿拉伯语等。  
更多信息请参见 [模型卡片](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct)、[博客](https://qwenlm.github.io/blog/qwen2.5/)、[GitHub](https://github.com/QwenLM/Qwen2.5) 和 [文档](https://qwen.readthedocs.io/en/latest/)。
* **qwen-7b-chat** - Qwen-7B是阿里巴巴云提出的Qwen（全称通义千问）大型语言模型系列的70亿参数版本。Qwen-7B是一个基于Transformer的大型语言模型，预训练在大量数据上，包括网络文本、书籍、代码等。更多关于Qwen的信息，请参考 [GitHub](https://github.com/QwenLM/Qwen) 代码库。
* **mpt-7b-chat** - MPT-7B是MosaicPretrainedTransformer (MPT) 模型家族的一部分，采用修改后的Transformer架构，优化了高效训练和推理。这些架构变化包括性能优化的层实现，以及通过注意力线性偏置替换位置嵌入来消除上下文长度限制（[ALiBi](https://arxiv.org/abs/2108.12409)）。由于这些改进，MPT模型可以高效训练并稳定收敛。MPT-7B-chat是一个用于对话生成的类聊天机器人模型，通过在[ShareGPT-Vicuna](https://huggingface.co/datasets/jeffwan/sharegpt_vicuna)、[HC3](https://huggingface.co/datasets/Hello-SimpleAI/HC3)、[Alpaca](https://huggingface.co/datasets/tatsu-lab/alpaca)、[HH-RLHF](https://huggingface.co/datasets/Anthropic/hh-rlhf) 和 [Evol-Instruct](https://huggingface.co/datasets/victor123/evol_instruct_70k) 数据集上微调而成。更多模型详情请参见 [博客文章](https://www.mosaicml.com/blog/mpt-7b)、[仓库](https://github.com/mosaicml/llm-foundry/) 和 [HuggingFace模型卡片](https://huggingface.co/mosaicml/mpt-7b-chat)。
* **chatglm3-6b** - ChatGLM3-6B是ChatGLM系列最新的开源模型。在保留前两代的诸多优秀特性（如流畅对话和低部署门槛）的同时，ChatGLM3-6B采用了更丰富的训练数据集、更充分的训练步骤和更合理的训练策略。ChatGLM3-6B采用新设计的 [Prompt格式](https://github.com/THUDM/ChatGLM3/blob/main/PROMPT_en.md)，除了正常的多轮对话外，还可以处理其他任务。更多模型细节请参见 [模型卡片](https://huggingface.co/THUDM/chatglm3-6b)
* **mistral-7b** - Mistral-7B-v0.1大型语言模型（LLM）是一个具有70亿参数的预训练生成文本模型。更多模型详情请参见 [模型卡片](https://huggingface.co/mistralai/Mistral-7B-v0.1)、[论文](https://arxiv.org/abs/2310.06825) 和 [发布博客文章](https://mistral.ai/news/announcing-mistral-7b/)。
* **mistral-7B-Instruct-v0.3** - Mistral-7B-Instruct-v0.3是一个先进的大型语言模型（LLM），适用于多种语言理解和生成任务，它是Mistral-7B-v0.3的指令微调版本。更多模型详情请参见 [模型卡片](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3)。
>**注意**：运行带有演示的模型时，您需要接受许可协议。  
>您必须是🤗 Hugging Face Hub的注册用户。请访问 [HuggingFace模型卡片](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)，仔细阅读使用条款并点击接受按钮。您将需要使用访问令牌来运行以下代码。有关访问令牌的更多信息，请参考 [文档中的此部分](https://huggingface.co/docs/hub/security-tokens)。  
>您可以在笔记本环境中通过以下代码登录Hugging Face Hub：

```python
    ## 登录Hugging Face Hub以获取预训练模型访问权限

    from huggingface_hub import notebook_login, whoami

    try:
        whoami()
        print('授权令牌已提供')
    except OSError:
        notebook_login()
```

* **zephyr-7b-beta** - Zephyr是一系列训练成帮助助手的模型。Zephyr-7B-beta是该系列的第二个模型，是对[mistralai/Mistral-7B-v0.1](https://huggingface.co/mistralai/Mistral-7B-v0.1)的微调版本，使用公开的合成数据集混合训练，采用[直接偏好优化（DPO）](https://arxiv.org/abs/2305.18290)。更多模型详情请参见 [技术报告](https://arxiv.org/abs/2310.16944) 和 [HuggingFace模型卡片](https://huggingface.co/HuggingFaceH4/zephyr-7b-beta)。
* **neural-chat-7b-v3-1** - 使用Intel Gaudi微调的Mistral-7b模型。该模型在开源数据集[Open-Orca/SlimOrca](https://huggingface.co/datasets/Open-Orca/SlimOrca)上微调，并通过[直接偏好优化（DPO）算法](https://arxiv.org/abs/2305.18290)对齐。更多细节请参见 [模型卡片](https://huggingface.co/Intel/neural-chat-7b-v3-1) 和 [博客文章](https://medium.com/@NeuralCompressor/the-practice-of-supervised-finetuning-and-direct-preference-optimization-on-habana-gaudi2-a1197d8a3cd3)。
* **notus-7b-v1** - Notus是一系列使用[直接偏好优化（DPO）](https://arxiv.org/abs/2305.18290)及相关[RLHF](https://huggingface.co/blog/rlhf)技术微调的模型。此模型是第一个版本，基于zephyr-7b-sft使用DPO微调。采用数据优先的方法，Notus-7B-v1与Zephyr-7B-beta的区别仅在于用于dDPO的偏好数据集。提出的数据集创建方法有助于有效微调Notus-7b，使其在[AlpacaEval](https://tatsu-lab.github.io/alpaca_eval/)上超越Zephyr-7B-beta和Claude 2。更多模型详情请参见 [模型卡片](https://huggingface.co/argilla/notus-7b-v1)。
* **youri-7b-chat** - Youri-7b-chat是一个基于Llama2的模型。[Rinna Co., Ltd.](https://rinna.co.jp/) 对Llama2模型进行了进一步预训练，使用英语和日语数据集混合以提高日语任务能力。该模型在Hugging Face hub上公开发布。您可以在 [rinna/youri-7b-chat项目页面](https://huggingface.co/rinna/youri-7b) 找到详细信息。
* **baichuan2-7b-chat** - 百川2是Baichuan Intelligence公司推出的新一代大规模开源语言模型。它在高质量语料库（2.6万亿token）上训练，取得了与同尺寸权威中文和英文基准测试中最佳性能。
* **internlm2-chat-1.8b** - InternLM2是InternLM系列的第二代。相比上一代模型，它在推理、数学和编程等多个能力上都有显著提升。更多模型详情请参见 [模型仓库](https://huggingface.co/internlm)。
* **glm-4-9b-chat** - GLM-4-9B是智谱AI发布的GLM-4系列最新一代预训练模型的开源版本。在语义、数学、推理、代码和知识等数据集上的评估中，GLM-4-9B及其人类偏好对齐版本GLM-4-9B-Chat的表现优于Llama-3-8B。除了多轮对话外，GLM-4-9B-Chat还具备网页浏览、代码执行、自定义工具调用（Function Call）和长文本推理（支持高达128K上下文）等高级功能。更多模型详情请参见 [模型卡片](https://huggingface.co/THUDM/glm-4-9b-chat/blob/main/README_en.md)、[技术报告](https://arxiv.org/pdf/2406.12793) 和 [仓库](https://github.com/THUDM/GLM-4)。
* **DeepSeek-R1-Distill-Qwen-1.5B** - 使用[DeepSeek-R1](https://huggingface.co/deepseek-ai/DeepSeek-R1)生成的推理数据微调的Qwen2.5-1.5B。更多信息请参见 [模型卡片](https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B)。
* **DeepSeek-R1-Distill-Qwen-7B** - 使用[DeepSeek-R1](https://huggingface.co/deepseek-ai/DeepSeek-R1)生成的推理数据微调的Qwen2.5-7B。更多信息请参见 [模型卡片](https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B)。
* **DeepSeek-R1-Distill-Llama-8B** - 使用[DeepSeek-R1](https://huggingface.co/deepseek-ai/DeepSeek-R1)生成的推理数据微调的Llama-3.1-8B。更多信息请参见 [模型卡片](https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B)。
* **GLM-4-9B-0414** - GLM-4-32B-0414系列模型，具有320亿参数。其性能可与OpenAI的GPT系列和DeepSeek V3/R1系列相媲美。它还支持非常友好的本地部署功能。GLM-4-32B-Base-0414在15TB高质量数据上预训练，包括大量推理类型的合成数据。更多信息请参见 [模型卡片](https://huggingface.co/THUDM/GLM-4-9B-0414)。
* **GLM-Z1-32B-0414** - GLM-Z1-32B-0414是一个具有深度思考能力的推理模型。这是基于GLM-4-32B-0414通过冷启动、扩展强化学习及在数学、代码和逻辑任务上的进一步训练开发的。与基础模型相比，GLM-Z1-32B-0414显著提升了数学能力和解决复杂任务的能力。更多信息请参见 [模型卡片](https://huggingface.co/THUDM/GLM-Z1-9B-0414)。
* **Qwen3-0.6/1.7/4B/8B/14B** - Qwen3是Qwen系列最新的大型语言模型，提供全面的密集模型和专家混合（MoE）模型套件。基于训练数据、模型架构和优化技术的广泛进步，Qwen3在先前发布的Qwen2.5基础上实现了以下关键改进。更多信息请参见 [模型卡片](https://huggingface.co/Qwen/Qwen3-8B)。
    Qwen3-8B int4模型使用以下量化参数：
        * mode: **INT4_ASYM**
        * ratio: **1.0**
        * group_size: **128**
        * scale_estimation: **True**
        * dataset: **wikitext2**
    Qwen3-1.7B/4B int4模型使用以下量化参数：
        * mode: **INT4_ASYM**
        * ratio: **1.0**
        * group_size: **128**
        * quant_method: **AWQ**
        * scale_estimation: **True**
        * dataset: **wikitext2**
* **AFM-4.5B** - AFM-4.5B是由Arcee.ai开发的45亿参数指令微调模型，专为企业级性能设计，可在从云端到边缘的各种部署环境中运行。基础模型在8万亿token的数据集上训练，其中包括6.5万亿token的一般预训练数据和1.5万亿token的中期训练数据，特别关注数学推理和代码生成。预训练后，模型在高质量指令数据集上进行了监督微调。指令微调模型进一步通过可验证奖励的强化学习和人类偏好进行精炼。更多信息请参见 [模型卡片](https://huggingface.co/arcee-ai/AFM-4.5B)。
  </details>

In [3]:
from llm_config import SUPPORTED_LLM_MODELS
import ipywidgets as widgets

In [4]:
model_languages = list(SUPPORTED_LLM_MODELS)

model_language = widgets.Dropdown(
    options=model_languages,
    value=model_languages[0],
    description="Model Language:",
    disabled=False,
)

model_language

Dropdown(description='Model Language:', options=('English', 'Chinese', 'Japanese'), value='English')

In [5]:
model_ids = list(SUPPORTED_LLM_MODELS[model_language.value])

model_id = widgets.Dropdown(
    options=model_ids,
    value=model_ids[0],
    description="Model:",
    disabled=False,
)

model_id

Dropdown(description='Model:', options=('qwen2-0.5b-instruct', 'tiny-llama-1b-chat', 'qwen2-1.5b-instruct', 'g…

In [6]:
model_configuration = SUPPORTED_LLM_MODELS[model_language.value][model_id.value]
print(f"Selected model {model_id.value}")

Selected model qwen2-0.5b-instruct


## 使用 Optimum-CLI 工具转换模型
[返回顶部 ⬆️](#Table-of-contents:)

🤗 [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) 是 🤗 [Transformers](https://huggingface.co/docs/transformers/index) 和 [Diffusers](https://huggingface.co/docs/diffusers/index) 库与 OpenVINO 之间的接口，用于在 Intel 架构上加速端到端管道。它提供了一个易于使用的 CLI 接口，用于将模型导出为 [OpenVINO 中间表示（IR）](https://docs.openvino.ai/2024/documentation/openvino-ir-format.html) 格式。

以下命令展示了使用 `optimum-cli` 进行模型导出的基本命令：

```
optimum-cli export openvino --model <model_id_or_path> --task <task> <out_dir>
```

其中 `--model` 参数是来自 HuggingFace Hub 的模型 ID 或本地保存的模型目录（使用 `.save_pretrained` 方法保存），`--task` 是导出模型应解决的 [支持任务](https://huggingface.co/docs/optimum/exporters/task_manager) 之一。对于 LLM，任务应为 `text-generation-with-past`。如果模型初始化需要使用远程代码，还需额外添加 `--trust-remote-code` 标志。

## 压缩模型权重

[权重压缩](https://docs.openvino.ai/2024/openvino-workflow/model-optimization-guide/weight-compression.html) 算法旨在压缩模型的权重，可用于优化大型模型的模型占用空间和性能，尤其适用于权重大小远大于激活值大小的场景，例如大语言模型（LLM）。与INT8压缩相比，INT4压缩能进一步提升性能，但会带来轻微的预测质量下降。

### 使用 Optimum-CLI 进行权重压缩
[返回顶部 ⬆️](#Table-of-contents:)

在使用 CLI 导出模型时，可以通过设置 `--weight-format` 为 fp16、int8 或 int4，对线性层、卷积层和嵌入层应用 fp16、8位或4位权重压缩。这种优化方式可以减少内存占用和推理延迟。默认情况下，int8/int4 的量化方案为 [非对称量化](https://github.com/openvinotoolkit/nncf/blob/develop/docs/compression_algorithms/Quantization.md#asymmetric-quantization)，若要使用 [对称量化](https://github.com/openvinotoolkit/nncf/blob/develop/docs/compression_algorithms/Quantization.md#symmetric-quantization)，可添加 `--sym` 参数。

对于 INT4 量化，还可以指定以下参数：
- `--group-size` 参数定义量化所用的组大小，设置为 -1 时将执行按列量化。
- `--ratio` 参数控制 4 位和 8 位量化的比例。例如，设置为 0.9 表示 90% 的层将被量化为 int4，10% 的层将被量化为 int8。

较小的 group_size 和 ratio 值通常能提升精度，但会牺牲模型大小和推理延迟。

>**注意**：在 dGPU 上，INT4/INT8 压缩模型可能不会带来速度提升。

In [7]:
from IPython.display import Markdown, display

prepare_int4_model = widgets.Checkbox(
    value=True,
    description="Prepare INT4 model",
    disabled=False,
)
prepare_int8_model = widgets.Checkbox(
    value=False,
    description="Prepare INT8 model",
    disabled=False,
)
prepare_fp16_model = widgets.Checkbox(
    value=False,
    description="Prepare FP16 model",
    disabled=False,
)

display(prepare_int4_model)
display(prepare_int8_model)
display(prepare_fp16_model)

Checkbox(value=True, description='Prepare INT4 model')

Checkbox(value=False, description='Prepare INT8 model')

Checkbox(value=False, description='Prepare FP16 model')

### 使用AWQ进行权重压缩
[返回顶部 ⬆️](#Table-of-contents:)

[激活感知权重量化](https://arxiv.org/abs/2306.00978)（AWQ）是一种用于更精确INT4压缩的模型权重调优算法。它能略微提升压缩后大语言模型的生成质量，但需要在校准数据集上花费大量额外时间进行权重调优。我们使用[Wikitext](https://huggingface.co/datasets/Salesforce/wikitext)数据集的`wikitext-2-raw-v1/train`子集进行校准。

以下您可以启用AWQ，在模型导出时额外应用INT4精度。

>**注意**：应用AWQ需要大量内存和时间。

>**注意**：有可能模型中不存在可应用AWQ的匹配模式，此时将跳过该步骤。

In [8]:
enable_awq = widgets.Checkbox(
    value=False,
    description="Enable AWQ",
    disabled=not prepare_int4_model.value,
)
display(enable_awq)

Checkbox(value=False, description='Enable AWQ')

我们现在可以保存浮点数和压缩模型变体

In [9]:
from pathlib import Path

pt_model_id = model_configuration["model_id"]
pt_model_name = model_id.value.split("-")[0]
fp16_model_dir = Path(model_id.value) / "FP16"
int8_model_dir = Path(model_id.value) / "INT8_compressed_weights"
int4_model_dir = Path(model_id.value) / "INT4_compressed_weights"


def convert_to_fp16():
    if (fp16_model_dir / "openvino_model.xml").exists():
        return
    remote_code = model_configuration.get("remote_code", False)
    export_command_base = "optimum-cli export openvino --model {} --task text-generation-with-past --weight-format fp16".format(pt_model_id)
    if remote_code:
        export_command_base += " --trust-remote-code"
    export_command = export_command_base + " " + str(fp16_model_dir)
    display(Markdown("**Export command:**"))
    display(Markdown(f"`{export_command}`"))
    ! $export_command


def convert_to_int8():
    if (int8_model_dir / "openvino_model.xml").exists():
        return
    int8_model_dir.mkdir(parents=True, exist_ok=True)
    remote_code = model_configuration.get("remote_code", False)
    export_command_base = "optimum-cli export openvino --model {} --task text-generation-with-past --weight-format int8".format(pt_model_id)
    if remote_code:
        export_command_base += " --trust-remote-code"
    export_command = export_command_base + " " + str(int8_model_dir)
    display(Markdown("**Export command:**"))
    display(Markdown(f"`{export_command}`"))
    ! $export_command


def convert_to_int4():
    compression_configs = {
        "zephyr-7b-beta": {
            "sym": True,
            "group_size": 64,
            "ratio": 0.6,
        },
        "mistral-7b": {
            "sym": True,
            "group_size": 64,
            "ratio": 0.6,
        },
        "minicpm-2b-dpo": {
            "sym": True,
            "group_size": 64,
            "ratio": 0.6,
        },
        "gemma-2b-it": {
            "sym": True,
            "group_size": 64,
            "ratio": 0.6,
        },
        "notus-7b-v1": {
            "sym": True,
            "group_size": 64,
            "ratio": 0.6,
        },
        "neural-chat-7b-v3-1": {
            "sym": True,
            "group_size": 64,
            "ratio": 0.6,
        },
        "llama-2-chat-7b": {
            "sym": True,
            "group_size": 128,
            "ratio": 0.8,
        },
        "llama-3-8b-instruct": {
            "sym": True,
            "group_size": 128,
            "ratio": 0.8,
        },
        "llama-3.1-8b-instruct": {
            "sym": True,
            "group_size": 128,
            "ratio": 1.0,
        },
        "gemma-7b-it": {
            "sym": True,
            "group_size": 128,
            "ratio": 0.8,
        },
        "chatglm2-6b": {
            "sym": True,
            "group_size": 128,
            "ratio": 0.72,
        },
        "qwen-7b-chat": {"sym": True, "group_size": 128, "ratio": 0.6},
        "red-pajama-3b-chat": {
            "sym": False,
            "group_size": 128,
            "ratio": 0.5,
        },
        "qwen2.5-7b-instruct": {"sym": True, "group_size": 128, "ratio": 1.0},
        "qwen2.5-3b-instruct": {"sym": True, "group_size": 128, "ratio": 1.0},
        "qwen2.5-14b-instruct": {"sym": True, "group_size": 128, "ratio": 1.0},
        "qwen2.5-1.5b-instruct": {"sym": True, "group_size": 128, "ratio": 1.0},
        "qwen2.5-0.5b-instruct": {"sym": True, "group_size": 128, "ratio": 1.0},
        "default": {
            "sym": False,
        },
    }

    model_compression_params = compression_configs.get(model_id.value, compression_configs["default"])
    if (int4_model_dir / "openvino_model.xml").exists():
        return
    remote_code = model_configuration.get("remote_code", False)
    export_command_base = "optimum-cli export openvino --model {} --task text-generation-with-past --weight-format int4".format(pt_model_id)
    group_size = model_compression_params.get("group_size")
    ratio = model_compression_params.get("ratio")
    int4_compression_args = ""
    if group_size is not None:
        int4_compression_args += " --group-size {}".format(group_size)
    if ratio is not None:
        int4_compression_args += " --ratio {}".format(ratio)
    if model_compression_params["sym"]:
        int4_compression_args += " --sym"
    if enable_awq.value:
        int4_compression_args += " --awq --dataset wikitext2 --num-samples 128"
    export_command_base += int4_compression_args
    if remote_code:
        export_command_base += " --trust-remote-code"
    export_command = export_command_base + " " + str(int4_model_dir)
    display(Markdown("**Export command:**"))
    display(Markdown(f"`{export_command}`"))
    ! $export_command


if prepare_fp16_model.value:
    convert_to_fp16()
if prepare_int8_model.value:
    convert_to_int8()
if prepare_int4_model.value:
    convert_to_int4()

让我们比较不同压缩类型下的模型大小

In [10]:
fp16_weights = fp16_model_dir / "openvino_model.bin"
int8_weights = int8_model_dir / "openvino_model.bin"
int4_weights = int4_model_dir / "openvino_model.bin"

if fp16_weights.exists():
    print(f"Size of FP16 model is {fp16_weights.stat().st_size / 1024 / 1024:.2f} MB")
for precision, compressed_weights in zip([8, 4], [int8_weights, int4_weights]):
    if compressed_weights.exists():
        print(f"Size of model with INT{precision} compressed weights is {compressed_weights.stat().st_size / 1024 / 1024:.2f} MB")
    if compressed_weights.exists() and fp16_weights.exists():
        print(f"Compression rate for INT{precision} model: {fp16_weights.stat().st_size / compressed_weights.stat().st_size:.3f}")

Size of model with INT4 compressed weights is 358.86 MB


## 选择推理设备和模型变体
[返回顶部 ⬆️](#Table-of-contents:)

>**注意**：在dGPU上，INT4/INT8压缩模型可能没有加速效果。

In [11]:
import requests

r = requests.get(
    url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
)
open("notebook_utils.py", "w").write(r.text)

from notebook_utils import device_widget

device = device_widget("CPU", exclude=["NPU"])

device

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("llm-chatbot.ipynb")

Dropdown(description='Device:', options=('CPU', 'AUTO'), value='CPU')

以下单元格演示了如何根据选定的模型权重变体和推理设备实例化模型

In [12]:
available_models = []
if int4_model_dir.exists():
    available_models.append("INT4")
if int8_model_dir.exists():
    available_models.append("INT8")
if fp16_model_dir.exists():
    available_models.append("FP16")

model_to_run = widgets.Dropdown(
    options=available_models,
    value=available_models[0],
    description="Model to run:",
    disabled=False,
)

model_to_run

Dropdown(description='Model to run:', options=('INT4',), value='INT4')

## 使用 Optimum Intel 实例化模型
[返回顶部 ⬆️](#Table-of-contents:)

Optimum Intel 可用于从 [Hugging Face Hub](https://huggingface.co/docs/optimum/intel/hf.co/models) 加载优化后的模型，并使用 Hugging Face API 创建管道，通过 OpenVINO Runtime 运行推理。Optimum 推理模型与 Hugging Face Transformers 模型 API 兼容。这意味着我们只需将 `AutoModelForXxx` 类替换为相应的 `OVModelForXxx` 类即可。

以下是 RedPajama 模型的示例：

```diff
-from transformers import AutoModelForCausalLM
+from optimum.intel.openvino import OVModelForCausalLM
from transformers import AutoTokenizer, pipeline

model_id = "togethercomputer/RedPajama-INCITE-Chat-3B-v1"
-model = AutoModelForCausalLM.from_pretrained(model_id)
+model = OVModelForCausalLM.from_pretrained(model_id, export=True)
```

模型类的初始化从调用 `from_pretrained` 方法开始。在下载和转换 Transformers 模型时，应添加参数 `export=True`（由于我们已提前转换过模型，因此无需提供此参数）。我们可以使用 `save_pretrained` 方法保存转换后的模型，以便下次使用。分词器类和管道 API 与 Optimum 模型兼容。

有关使用 HuggingFace Optimum API 进行 OpenVINO LLM 推理的更多详细信息，请参阅 [LLM 推理指南](https://docs.openvino.ai/2025/openvino-workflow-generative.html)。

In [13]:
from transformers import AutoConfig, AutoTokenizer
from optimum.intel.openvino import OVModelForCausalLM

import openvino as ov
import openvino.properties as props
import openvino.properties.hint as hints
import openvino.properties.streams as streams


if model_to_run.value == "INT4":
    model_dir = int4_model_dir
elif model_to_run.value == "INT8":
    model_dir = int8_model_dir
else:
    model_dir = fp16_model_dir
print(f"Loading model from {model_dir}")

ov_config = {hints.performance_mode(): hints.PerformanceMode.LATENCY, streams.num(): "1", props.cache_dir(): ""}

if "GPU" in device.value and "qwen2-7b-instruct" in model_id.value:
    ov_config["GPU_ENABLE_SDPA_OPTIMIZATION"] = "NO"

# On a GPU device a model is executed in FP16 precision. For red-pajama-3b-chat model there known accuracy
# issues caused by this, which we avoid by setting precision hint to "f32".
core = ov.Core()

if model_id.value == "red-pajama-3b-chat" and "GPU" in core.available_devices and device.value in ["GPU", "AUTO"]:
    ov_config["INFERENCE_PRECISION_HINT"] = "f32"

model_name = model_configuration["model_id"]
tok = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)

ov_model = OVModelForCausalLM.from_pretrained(
    model_dir,
    device=device.value,
    ov_config=ov_config,
    config=AutoConfig.from_pretrained(model_dir, trust_remote_code=True),
    trust_remote_code=True,
)

Loading model from qwen2-0.5b-instruct/INT4_compressed_weights


Compiling the model to CPU ...


In [14]:
tokenizer_kwargs = model_configuration.get("tokenizer_kwargs", {})
test_string = "2 + 2 ="
input_tokens = tok(test_string, return_tensors="pt", **tokenizer_kwargs)
answer = ov_model.generate(**input_tokens, max_new_tokens=2)
print(tok.batch_decode(answer, skip_special_tokens=True)[0])

2 + 2 = 4


## 运行聊天机器人
[返回顶部 ⬆️](#Table-of-contents:)

现在，当模型创建完成后，我们可以使用 [Gradio](https://www.gradio.app/) 设置聊天机器人界面。  
下图展示了聊天机器人管道的工作原理：

![生成管道](https://user-images.githubusercontent.com/29454499/255523209-d9336491-c7ba-4dc1-98f0-07f23743ce89.png)

如图所示，该管道与指令跟随（instruction-following）非常相似，唯一的区别是，将之前的对话历史作为输入，与用户的下一个问题一起传递，以获得更广泛的上下文。在第一次迭代中，用户提供的指令会与现有的对话历史合并，然后通过分词器转换为 token id，再将准备好的输入提供给模型。模型以 logits 格式生成所有 token 的概率。下一个 token 的选择方式由所选的解码方法决定。你可以在本 [博客](https://huggingface.co/blog/how-to-generate) 中找到关于最流行解码方法的更多信息。生成结果会更新对话历史，以供下一次对话使用。这使得后续问题与之前提供的内容有更强的关联性，并允许用户对之前给出的答案进行澄清。https://docs.openvino.ai/2025/openvino-workflow-generative.html

有多个参数可以控制文本生成质量：  
  * `Temperature` 是一个用于控制 AI 生成文本创意程度的参数。通过调整 `temperature`，可以影响 AI 模型的概率分布，使生成的文本更集中或更多样化。  
  请看以下示例：AI 模型需要完成句子 “The cat is ____.”，其 token 概率如下：  

    playing: 0.5  
    sleeping: 0.25  
    eating: 0.15  
    driving: 0.05  
    flying: 0.05  

    - **低温度**（例如 0.2）：AI 模型变得更加集中和确定性，选择概率最高的 token，如 “playing”。  
    - **中等温度**（例如 1.0）：AI 模型在创意和集中度之间保持平衡，根据 token 的概率进行选择，无明显偏倚，如 “playing”、“sleeping” 或 “eating”。  
    - **高温度**（例如 2.0）：AI 模型变得更加冒险，增加选择不太可能 token 的概率，如 “driving” 和 “flying”。  
  * `Top-p`，也称为核采样（nucleus sampling），是一个根据 token 的累积概率来控制 AI 模型考虑 token 范围的参数。通过调整 `top-p` 值，可以影响 AI 模型的 token 选择，使其更集中或更多样化。  
  使用相同的猫的例子，考虑以下 `top_p` 设置：  
    - **低 top_p**（例如 0.5）：AI 模型仅考虑累积概率最高的 token，如 “playing”。  
    - **中等 top_p**（例如 0.8）：AI 模型考虑累积概率较高的 token，如 “playing”、“sleeping” 和 “eating”。  
    - **高 top_p**（例如 1.0）：AI 模型考虑所有 token，包括概率较低的，如 “driving” 和 “flying”。  
  * `Top-k` 是另一种流行的采样策略。与 Top-P（从累积概率超过 P 的最小词集选择）不同，Top-K 采样会过滤出最可能的 K 个下一个词，并将概率质量重新分配给这 K 个词。在我们的猫的例子中，如果 k=3，则仅考虑 “playing”、“sleeping” 和 “eating” 作为可能的下一个词。  
  * `Repetition Penalty` 该参数可以根据 token 在文本（包括输入提示）中出现的频率对其进行惩罚。已经出现五次的 token 会比仅出现一次的 token 受到更重的惩罚。值为 1 表示无惩罚，大于 1 的值会抑制重复 token 的出现。https://docs.openvino.ai/2025/openvino-workflow-generative.html

In [ ]:
import torch
from threading import Event, Thread

from transformers import (
    AutoTokenizer,
    StoppingCriteria,
    StoppingCriteriaList,
    TextIteratorStreamer,
)


model_name = model_configuration["model_id"]
start_message = model_configuration["start_message"]
history_template = model_configuration.get("history_template")
has_chat_template = model_configuration.get("has_chat_template", history_template is None)
current_message_template = model_configuration.get("current_message_template")
stop_tokens = model_configuration.get("stop_tokens")
tokenizer_kwargs = model_configuration.get("tokenizer_kwargs", {})

max_new_tokens = 256


class StopOnTokens(StoppingCriteria):
    def __init__(self, token_ids):
        self.token_ids = token_ids

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        for stop_id in self.token_ids:
            if input_ids[0][-1] == stop_id:
                return True
        return False


if stop_tokens is not None:
    if isinstance(stop_tokens[0], str):
        stop_tokens = tok.convert_tokens_to_ids(stop_tokens)

    stop_tokens = [StopOnTokens(stop_tokens)]


def default_partial_text_processor(partial_text: str, new_text: str):
    """
    helper for updating partially generated answer, used by default

    Params:
      partial_text: text buffer for storing previosly generated text
      new_text: text update for the current step
    Returns:
      updated text string

    """
    partial_text += new_text
    return partial_text


text_processor = model_configuration.get("partial_text_processor", default_partial_text_processor)


def convert_history_to_token(history: list[tuple[str, str]]):
    """
    function for conversion history stored as list pairs of user and assistant messages to tokens according to model expected conversation template
    Params:
      history: dialogue history
    Returns:
      history in token format
    """
    if pt_model_name == "baichuan2":
        system_tokens = tok.encode(start_message)
        history_tokens = []
        for old_query, response in history[:-1]:
            round_tokens = []
            round_tokens.append(195)
            round_tokens.extend(tok.encode(old_query))
            round_tokens.append(196)
            round_tokens.extend(tok.encode(response))
            history_tokens = round_tokens + history_tokens
        input_tokens = system_tokens + history_tokens
        input_tokens.append(195)
        input_tokens.extend(tok.encode(history[-1][0]))
        input_tokens.append(196)
        input_token = torch.LongTensor([input_tokens])
    elif history_template is None or has_chat_template:
        messages = [{"role": "system", "content": start_message}]
        for idx, (user_msg, model_msg) in enumerate(history):
            if idx == len(history) - 1 and not model_msg:
                messages.append({"role": "user", "content": user_msg})
                break
            if user_msg:
                messages.append({"role": "user", "content": user_msg})
            if model_msg:
                messages.append({"role": "assistant", "content": model_msg})

        input_token = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_tensors="pt")
    else:
        text = start_message + "".join(
            ["".join([history_template.format(num=round, user=item[0], assistant=item[1])]) for round, item in enumerate(history[:-1])]
        )
        text += "".join(
            [
                "".join(
                    [
                        current_message_template.format(
                            num=len(history) + 1,
                            user=history[-1][0],
                            assistant=history[-1][1],
                        )
                    ]
                )
            ]
        )
        input_token = tok(text, return_tensors="pt", **tokenizer_kwargs).input_ids
    return input_token


def bot(history, temperature, top_p, top_k, repetition_penalty, conversation_id):
    """
    callback function for running chatbot on submit button click

    Params:
      history: conversation history
      temperature:  parameter for control the level of creativity in AI-generated text.
                    By adjusting the `temperature`, you can influence the AI model's probability distribution, making the text more focused or diverse.
      top_p: parameter for control the range of tokens considered by the AI model based on their cumulative probability.
      top_k: parameter for control the range of tokens considered by the AI model based on their cumulative probability, selecting number of tokens with highest probability.
      repetition_penalty: parameter for penalizing tokens based on how frequently they occur in the text.
      conversation_id: unique conversation identifier.

    """

    # Construct the input message string for the model by concatenating the current system message and conversation history
    # Tokenize the messages string
    input_ids = convert_history_to_token(history)
    if input_ids.shape[1] > 2000:
        history = [history[-1]]
        input_ids = convert_history_to_token(history)
    streamer = TextIteratorStreamer(tok, timeout=3600.0, skip_prompt=True, skip_special_tokens=True)
    generate_kwargs = dict(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=temperature > 0.0,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=repetition_penalty,
        streamer=streamer,
    )
    if stop_tokens is not None:
        generate_kwargs["stopping_criteria"] = StoppingCriteriaList(stop_tokens)

    stream_complete = Event()

    def generate_and_signal_complete():
        """
        genration function for single thread
        """
        ov_model.generate(**generate_kwargs)
        stream_complete.set()

    t1 = Thread(target=generate_and_signal_complete)
    t1.start()

    # Initialize an empty string to store the generated text
    partial_text = ""
    for new_text in streamer:
        partial_text = text_processor(partial_text, new_text)
        history[-1][1] = partial_text
        yield history


def request_cancel():
    ov_model.request.cancel()

In [ ]:
if not Path("gradio_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/llm-chatbot/gradio_helper.py")
    open("gradio_helper.py", "w").write(r.text)

from gradio_helper import make_demo

demo = make_demo(run_fn=bot, stop_fn=request_cancel, title=f"OpenVINO {model_id.value} Chatbot", language=model_language.value)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)
# If you are launching remotely, specify server_name and server_port
# EXAMPLE: `demo.launch(server_name='your server name', server_port='server port in int')`
# To learn more please refer to the Gradio docs: https://gradio.app/docs/

### 下一步

除了聊天机器人，我们还可以使用 LangChain 通过附加数据来增强大语言模型（LLM）的知识，从而构建能够对私有数据或模型截止日期之后引入的数据进行推理的人工智能应用。你可以在 [检索增强生成（RAG）示例](../llm-rag-langchain/) 中找到该解决方案。